# Analyse du Data Drift

In [3]:
import pandas as pd
from sklearn import datasets

from evidently.future.datasets import Dataset
from evidently.future.datasets import DataDefinition

from evidently.future.report import Report
from evidently.future.metrics import *
from evidently.future.presets import *
from evidently.future.tests import *

# Data

In [8]:
domain_features=['DAYS_BIRTH', 'CODE_GENDER_F', 'FLAG_RISKED_ORGANIZATION_TYPE', 
                 'RATIO_INCOME_TO_FAM_MEMBERS', 'FLAG_STABLE_INCOME_TYPE',
                 'EXT_SOURCE_2',
                 'AMT_ALL_SUM_MEAN', 'RATIO_ALL_SUM_DEBT_TO_INCOME', 'RATIO_ALL_SUM_OVERDUE_TO_INCOME', 'RATIO_ALL_MAX_OVERDUE_TO_INCOME']

In [9]:
data_ref = pd.read_csv('C:/Users/innov/Python_venv/OC-PJ7/input/application_train_20250103.csv', usecols=domain_features)
data_ref.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 307510 entries, 0 to 307509
Data columns (total 10 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   DAYS_BIRTH                       307510 non-null  int64  
 1   EXT_SOURCE_2                     307510 non-null  float64
 2   RATIO_INCOME_TO_FAM_MEMBERS      307510 non-null  float64
 3   FLAG_STABLE_INCOME_TYPE          307510 non-null  int64  
 4   FLAG_RISKED_ORGANIZATION_TYPE    307510 non-null  int64  
 5   CODE_GENDER_F                    307510 non-null  int64  
 6   AMT_ALL_SUM_MEAN                 307510 non-null  float64
 7   RATIO_ALL_SUM_OVERDUE_TO_INCOME  307510 non-null  float64
 8   RATIO_ALL_MAX_OVERDUE_TO_INCOME  307510 non-null  float64
 9   RATIO_ALL_SUM_DEBT_TO_INCOME     307510 non-null  float64
dtypes: float64(6), int64(4)
memory usage: 25.8 MB


In [11]:
data_prod = pd.read_csv('C:/Users/innov/Python_venv/OC-PJ7/input/application_test_20250103.csv', usecols=domain_features)
data_prod.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48744 entries, 0 to 48743
Data columns (total 10 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   DAYS_BIRTH                       48744 non-null  int64  
 1   EXT_SOURCE_2                     48744 non-null  float64
 2   RATIO_INCOME_TO_FAM_MEMBERS      48744 non-null  float64
 3   FLAG_STABLE_INCOME_TYPE          48744 non-null  int64  
 4   FLAG_RISKED_ORGANIZATION_TYPE    48744 non-null  int64  
 5   CODE_GENDER_F                    48744 non-null  int64  
 6   AMT_ALL_SUM_MEAN                 48744 non-null  float64
 7   RATIO_ALL_SUM_OVERDUE_TO_INCOME  48744 non-null  float64
 8   RATIO_ALL_MAX_OVERDUE_TO_INCOME  48744 non-null  float64
 9   RATIO_ALL_SUM_DEBT_TO_INCOME     48744 non-null  float64
dtypes: float64(6), int64(4)
memory usage: 3.7 MB


Création dataset pour Evidently

In [12]:
schema = DataDefinition(
    numerical_columns=['DAYS_BIRTH', 'RATIO_INCOME_TO_FAM_MEMBERS', 'EXT_SOURCE_2', 'AMT_ALL_SUM_MEAN', 
                       'RATIO_ALL_SUM_DEBT_TO_INCOME', 'RATIO_ALL_SUM_OVERDUE_TO_INCOME', 'RATIO_ALL_MAX_OVERDUE_TO_INCOME'],
    categorical_columns=['CODE_GENDER_F', 'FLAG_RISKED_ORGANIZATION_TYPE', 'FLAG_STABLE_INCOME_TYPE'],
    )

In [28]:
current = Dataset.from_pandas(
    pd.DataFrame(data_prod),
    data_definition=schema
)

In [29]:
reference = Dataset.from_pandas(
    pd.DataFrame(data_ref),
    data_definition=schema
)

# Report DataSummary Dataset

In [35]:
report = Report([
    DataSummaryPreset()
],
include_tests="True")
my_eval = report.run(current, reference)
my_eval.save_html('./reports/DataSummaryPreset.html')

In [36]:
report = Report(metrics=[
    DataDriftPreset(), 
])

my_eval = report.run(reference_data=reference, current_data=current)
my_eval.save_html('./reports/DataDriftPreset.html')